# CLI Demo — Medha v0.4.0

This notebook walks through every `medha` CLI command against an **in-memory backend** —
no external services required.

| Command | Purpose |
|---|---|
| `medha stats` | Entry counts and backend type |
| `medha warm FILE` | Bulk-load entries from JSON / JSONL |
| `medha expire` | Delete entries past their TTL |
| `medha dedup` | Remove entries sharing the same query hash |
| `medha invalidate QUESTION` | Remove one entry by question text |
| `medha export` | Dump cache to CSV or JSON |
| `medha feedback QUESTION` | Record correct / incorrect signal |

> **`InMemoryBackend` is per-process.** Each `!medha` shell invocation starts with an
> empty backend, so `stats`, `expire`, `invalidate`, and `export` CLI cells show
> zero / empty results. The `warm` command is the exception — its count is captured
> within one process. Python API cells in sections 6–8 demonstrate the same operations
> in-process with real data so you can see non-trivial output.
>
> With a persistent backend (`MEDHA_BACKEND_TYPE=qdrant`) all commands share live data.

**Requirements:** `pip install "medha-archai[cli,fastembed]"`

## 1 — Installation

Install the CLI extra and FastEmbed (required for `medha warm`).
FastEmbed downloads a ~40 MB model on first use; subsequent runs hit the local cache.

In [ ]:
!pip install "medha-archai[cli,fastembed]"

## 2 — Verify install

`medha --help` confirms the CLI is on PATH and lists every available sub-command
with its description.

In [ ]:
!medha --help

## 3 — Prepare data

Store five Text-to-SQL question–query pairs programmatically and save the collection
to a JSONL file. Each line is a JSON object with at least `question` and
`generated_query` keys; `response_summary` is optional.

`medha warm` reads this exact format. The file is the portable artefact that lets
subsequent CLI calls reload the data into any backend.

In [ ]:
import io
import json
import os
import tempfile

from medha import Medha, Settings
from medha.embeddings.fastembed_adapter import FastEmbedAdapter

# Shared embedder — reused in later Python API demo cells
embedder = FastEmbedAdapter(model_name='BAAI/bge-small-en-v1.5', cache_dir='.fastembed_cache')

ENTRIES = [
    {
        'question': 'How many users are registered?',
        'generated_query': 'SELECT COUNT(*) FROM users',
        'response_summary': 'Total registered users',
    },
    {
        'question': 'What is the total revenue this year?',
        'generated_query': 'SELECT SUM(amount) FROM invoices WHERE YEAR(created_at) = YEAR(NOW())',
        'response_summary': 'Annual revenue figure',
    },
    {
        'question': 'Show the top 10 most expensive products',
        'generated_query': 'SELECT * FROM products ORDER BY price DESC LIMIT 10',
        'response_summary': 'Priciest products',
    },
    {
        'question': 'Count orders placed today',
        'generated_query': 'SELECT COUNT(*) FROM orders WHERE DATE(created_at) = CURDATE()',
        'response_summary': "Today's order volume",
    },
    {
        'question': 'What is the average employee salary?',
        'generated_query': 'SELECT AVG(salary) FROM employees',
        'response_summary': 'Mean salary across all employees',
    },
]

ENTRIES_FILE = os.path.join(tempfile.gettempdir(), 'medha_demo_entries.jsonl')
with open(ENTRIES_FILE, 'w', encoding='utf-8') as f:
    for entry in ENTRIES:
        f.write(json.dumps(entry) + '\n')

print(f'Saved {len(ENTRIES)} entries → {ENTRIES_FILE}')
print()
print('First entry (JSONL format):')
print(json.dumps(ENTRIES[0], indent=2))

## 4 — `medha stats`

`stats` reports structural information about a collection:
- Collection name and backend type
- Entry count in the main collection
- Entry count in the template sub-collection

Performance metrics (hit rate, latency percentiles) are **not** available from the CLI:
`CacheStats` is a non-persistent in-process accumulator on the `Medha` object.

> With `InMemoryBackend`, each fresh CLI call sees 0 entries.
> With `MEDHA_BACKEND_TYPE=qdrant` the command returns the live count from Qdrant.

In [ ]:
%env MEDHA_BACKEND_TYPE=memory
!medha stats

## 5 — `medha warm`

`warm FILE` reads a `.json` or `.jsonl` file and calls `store_many()` internally,
embedding every question in batches.

**Requires a real embedder.** Set `MEDHA_EMBEDDER_TYPE` to one of:
`fastembed`, `openai`, `cohere`, `gemini`. The default `_noop` raises an error.

Expected output:
```
Progress: 5/5 entries stored.
Warmed 5 entries into 'default'.
```

The progress line fires once per batch. The final line shows the total loaded
within this single CLI process (before = 0, after = 5 for InMemoryBackend).

In [ ]:
%env MEDHA_BACKEND_TYPE=memory
%env MEDHA_EMBEDDER_TYPE=fastembed
!medha warm {ENTRIES_FILE}

## 6 — `medha expire` and `medha dedup`

### `medha expire`
Deletes every entry whose TTL has elapsed. Warm entries with `--ttl N` to give
them a finite lifespan; run `medha expire` periodically to purge stale rows.

### `medha dedup`
Removes entries sharing the same `query_hash` (derived from the generated query
string), keeping the most-recently stored entry per hash. Requires `pandas`.

Both commands print the count of removed entries.

> With `InMemoryBackend` each CLI call starts with an empty backend, so both
> commands report 0 here. The Python API cells below demonstrate the same operations
> in-process with real data.

In [ ]:
# CLI form — reports 0 on a fresh InMemoryBackend
%env MEDHA_BACKEND_TYPE=memory
!medha expire

In [ ]:
# Python API equivalent: store 3 entries with a 1-second TTL, wait, then expire
import asyncio

async with Medha('expire_demo', embedder=embedder, settings=Settings(backend_type='memory')) as cache:
    for e in ENTRIES[:3]:
        await cache.store(e['question'], e['generated_query'], ttl=1)
    count_before = await cache._backend.count('expire_demo')
    await asyncio.sleep(1.1)  # let TTLs elapse
    removed = await cache.expire()
    count_after = await cache._backend.count('expire_demo')

print(f'expire() — stored: {count_before}, removed: {removed}, remaining: {count_after}')
print()
print('CLI equivalent (persistent backend):')
print('  MEDHA_BACKEND_TYPE=qdrant medha warm entries.jsonl --ttl 60')
print('  # ... wait for TTLs to elapse ...')
print('  MEDHA_BACKEND_TYPE=qdrant medha expire')

In [ ]:
# CLI form — reports 0 on a fresh InMemoryBackend
%env MEDHA_BACKEND_TYPE=memory
!medha dedup

In [ ]:
# Python API equivalent: three questions → same SQL → same query_hash → duplicates
async with Medha('dedup_demo', embedder=embedder, settings=Settings(backend_type='memory')) as cache:
    for question in ['Count users', 'How many users?', 'Total user count']:
        await cache.store(question, 'SELECT COUNT(*) FROM users')
    await cache.store('List products', 'SELECT * FROM products')  # different SQL — survives

    count_before = await cache._backend.count('dedup_demo')
    removed = await cache.dedup_collection(strategy='keep_latest')
    count_after = await cache._backend.count('dedup_demo')

print(f'dedup_collection() — before: {count_before}, removed: {removed}, after: {count_after}')
print()
print('CLI equivalent (persistent backend):')
print('  MEDHA_BACKEND_TYPE=qdrant medha dedup')

## 7 — `medha invalidate`

`invalidate QUESTION` removes the entry whose normalised question matches the
argument. Prints `"Removed."` on success or `"Not found."` if no entry matches.

Uses the default `_NoOpEmbedder` — no real embedder needed because invalidation
is a plain text lookup on the normalised question.

> `InMemoryBackend` starts empty, so `"Not found."` is expected here.
> The Python API cell below shows a successful removal from a pre-loaded cache.

In [ ]:
# CLI form — "Not found." because the backend starts empty
%env MEDHA_BACKEND_TYPE=memory
!medha invalidate "How many users are registered?"

In [ ]:
# Python API equivalent: populate cache, then invalidate one entry
target = 'How many users are registered?'

async with Medha('inv_demo', embedder=embedder, settings=Settings(backend_type='memory')) as cache:
    for e in ENTRIES:
        await cache.store(e['question'], e['generated_query'])
    count_before = await cache._backend.count('inv_demo')
    removed = await cache.invalidate(target)
    count_after = await cache._backend.count('inv_demo')

print(f"invalidate('{target}')")
print(f'  removed : {removed}')
print(f'  before  : {count_before} entries')
print(f'  after   : {count_after} entries')

## 8 — `medha export --format csv`

`export` serialises every entry in the collection to CSV (default) or JSON.
Use `--output FILE` to write to a file; omit it to print to stdout.

```bash
medha export --format csv                       # stdout
medha export --format csv --output cache.csv    # file
medha export --format json --output cache.json  # JSON records
```

Requires `pandas` (`pip install pandas`).

The CLI cell below captures stdout and parses it with pandas. The result is an
empty DataFrame because `InMemoryBackend` starts fresh. The Python API cell that
follows populates a cache in-process and shows the export with real rows.

In [ ]:
import pandas as pd

# CLI form — empty because InMemoryBackend starts fresh
%env MEDHA_BACKEND_TYPE=memory
csv_lines = !medha export --format csv
df_cli = pd.read_csv(io.StringIO('\n'.join(csv_lines)))
print(f'Rows from CLI export: {len(df_cli)}  (0 — ephemeral backend)')
df_cli

In [ ]:
# Python API equivalent: populate cache in-process, then export
async with Medha('export_demo', embedder=embedder, settings=Settings(backend_type='memory')) as cache:
    for e in ENTRIES:
        await cache.store(
            e['question'],
            e['generated_query'],
            response_summary=e.get('response_summary'),
        )
    df = await cache.export_to_dataframe()

# Save to CSV — what medha export --format csv --output cache.csv would produce
EXPORT_FILE = os.path.join(tempfile.gettempdir(), 'medha_export.csv')
df.to_csv(EXPORT_FILE, index=False)
print(f'Exported {len(df)} entries → {EXPORT_FILE}')
print(f'CLI equivalent: medha export --format csv --output {EXPORT_FILE}')
print()

In [ ]:
# Display selected columns from the exported DataFrame
df[['original_question', 'generated_query', 'response_summary', 'usage_count']]

## 9 — Environment variable reference

All `MEDHA_*` variables are read at CLI startup via `pydantic-settings`.
Pass them as shell env vars or store them in a `.env` file in the working directory.

| Variable | Default | Commands | Description |
|---|---|---|---|
| `MEDHA_BACKEND_TYPE` | `memory` | all | Backend: `memory`, `qdrant`, `pgvector`, `elasticsearch`, `chroma`, … |
| `MEDHA_COLLECTION` | `default` | all | Collection name (overridden by `--collection`) |
| `MEDHA_EMBEDDER_TYPE` | `_noop` | `warm` | Embedder: `fastembed`, `openai`, `cohere`, `gemini`, `_noop` |
| `MEDHA_FASTEMBED_MODEL` | `BAAI/bge-small-en-v1.5` | `warm` | FastEmbed model identifier |
| `MEDHA_QDRANT_HOST` | `localhost` | all (Qdrant) | Qdrant server host |
| `MEDHA_QDRANT_PORT` | `6333` | all (Qdrant) | Qdrant gRPC port |
| `MEDHA_QDRANT_URL` | — | all (Qdrant) | Full Qdrant URL (overrides host + port) |
| `MEDHA_QDRANT_API_KEY` | — | all (Qdrant) | Qdrant Cloud API key |
| `OPENAI_API_KEY` | — | `warm` | OpenAI key when `MEDHA_EMBEDDER_TYPE=openai` |
| `COHERE_API_KEY` | — | `warm` | Cohere key when `MEDHA_EMBEDDER_TYPE=cohere` |
| `GOOGLE_API_KEY` | — | `warm` | Gemini key when `MEDHA_EMBEDDER_TYPE=gemini` |
| `MEDHA_DEFAULT_TTL_SECONDS` | — | `warm` | Default TTL per entry (seconds); omit for immortal entries |
| `MEDHA_FEEDBACK_INCORRECT_THRESHOLD` | — | all | Auto-invalidate after N incorrect feedback signals |
| `MEDHA_BATCH_SIZE` | `100` | `warm` | Entries per upsert batch |
| `MEDHA_SCORE_THRESHOLD_SEMANTIC` | `0.85` | `warm` | Semantic similarity threshold |

> **Cloud embedder keys** are read from the provider's standard env vars
> (`OPENAI_API_KEY`, `COHERE_API_KEY`, `GOOGLE_API_KEY`) rather than `MEDHA_*` prefixed
> vars, so you don’t need to duplicate secrets under a second name.

### Quick-start `.env` for a Qdrant deployment

```ini
MEDHA_BACKEND_TYPE=qdrant
MEDHA_QDRANT_URL=https://xyz.qdrant.tech
MEDHA_QDRANT_API_KEY=your-api-key
MEDHA_COLLECTION=prod_cache
MEDHA_EMBEDDER_TYPE=fastembed
MEDHA_FASTEMBED_MODEL=BAAI/bge-small-en-v1.5
MEDHA_DEFAULT_TTL_SECONDS=86400
```

With this file in place, all `medha` commands automatically connect to Qdrant and
share the same persistent collection.